# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state.pt")
validation_fraction = 0.10
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(6, 8),
    spatial_kernel_size_z=(1, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(12,),
    temporal_st_kernel_sizes=(3,),
    temporal_ts_channels=(8,),
    temporal_ts_kernel_sizes=(5,),
    spatial_agg_channels=(12,),
    spatial_agg_kernel_size_z=(3,),
    spatial_agg_kernel_size_xy=(3,),
    spatial_agg_stride_z=(1,),
    spatial_agg_stride_xy=(1,),
    spatial_agg_pool_kernel_z=(1,),
    spatial_agg_pool_kernel_xy=(2,),
    spatial_agg_pool_stride_z=(1,),
    spatial_agg_pool_stride_xy=(2,),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=8,
    num_prototypes=8,
    dropout=0.5,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=2e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)


In [ ]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


In [ ]:
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


In [ ]:
model.pretrain_history_.tail()